In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from collections import Counter
import xgboost as xgb

In [2]:
df = pd.read_csv(r"C:\Users\anubh\OneDrive\Desktop\AI-medical-assistant\data\diabetes\diabetes_prediction_dataset.csv")

df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


In [4]:
df.describe()

,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,diabetes
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,41.885856,0.07485,0.039420,27.320767,5.527507,138.058060,0.085000
std,22.516840,0.26315,0.194593,6.636783,1.070672,40.708136,0.278883
min,0.080000,0.00000,0.000000,10.010000,3.500000,80.000000,0.000000
25%,24.000000,0.00000,0.000000,23.630000,4.800000,100.000000,0.000000
50%,43.000000,0.00000,0.000000,27.320000,5.800000,140.000000,0.000000
75%,60.000000,0.00000,0.000000,29.580000,6.200000,159.000000,0.000000
max,80.000000,1.00000,1.000000,95.690000,9.000000,300.000000,1.000000


In [5]:
df['diabetes'].value_counts()

diabetes
0    91500
1     8500
Name: count, dtype: int64

In [6]:
le = LabelEncoder()
df["gender"]          = le.fit_transform(df["gender"])           # Male=1, Female=0
df["smoking_history"] = le.fit_transform(df["smoking_history"])  # encoded 0-5

In [7]:
df['smoking_history'].value_counts()

smoking_history
0    35816
4    35095
3     9352
1     9286
5     6447
2     4004
Name: count, dtype: int64

In [8]:
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

In [9]:
print("Original class distribution:")
print(Counter(y))   # {0: 91500, 1: 8500}

Original class distribution:
Counter({0: 91500, 1: 8500})


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # keeps the same ratio in both splits
)

In [11]:
print(f"\nTrain size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")
print("Train class distribution (before SMOTE):", Counter(y_train))


Train size: 80000  |  Test size: 20000
Train class distribution (before SMOTE): Counter({0: 73200, 1: 6800})


In [12]:
smote = SMOTE(
    sampling_strategy=0.5,   # after SMOTE: minority = 50% of majority
                             # gives ~30k positives vs ~73k negatives (≈1:2.4)
                             # a perfect 1:1 balance can hurt precision — 1:2 is a sweet spot
    random_state=42,
    k_neighbors=5
)

In [13]:
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [14]:
print("\nTrain class distribution (after SMOTE):", Counter(y_train_res))


Train class distribution (after SMOTE): Counter({0: 73200, 1: 36600})


In [15]:
neg = Counter(y_train_res)[0]
pos = Counter(y_train_res)[1]
spw = round(neg / pos, 2)
print(f"\nscale_pos_weight = {spw}  (neg/pos after SMOTE)")


scale_pos_weight = 2.0  (neg/pos after SMOTE)


In [16]:
# Model Building

model = xgb.XGBClassifier(
    n_estimators          = 500,
    max_depth             = 5,          # shallower = less overfit on majority
    learning_rate         = 0.05,       # slower learning = better generalisation
    scale_pos_weight      = spw,
    min_child_weight      = 5,          # prevents splits on tiny minority groups
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    gamma                 = 1,          # min loss reduction to make a split
    reg_alpha             = 0.1,        # L1 regularisation
    reg_lambda            = 1.5,        # L2 regularisation
    eval_metric           = "aucpr",    # AUC-PR is better than AUC-ROC for imbalanced data
    use_label_encoder     = False,
    random_state          = 42,
    early_stopping_rounds = 30,
)

In [17]:
model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test, y_test)],
    verbose=100
)

[0]	validation_0-aucpr:0.82998


C:\Users\anubh\AppData\Roaming\Python\Python312\site-packages\xgboost\callback.py:385: UserWarning: [21:52:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[100]	validation_0-aucpr:0.87005
[200]	validation_0-aucpr:0.88060
[300]	validation_0-aucpr:0.88215
[302]	validation_0-aucpr:0.88213


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=30,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [19]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, f1_score


In [20]:
y_pred_prob = model.predict_proba(X_test)[:, 1]
 
thresholds  = np.arange(0.20, 0.60, 0.01)
f1_scores   = [f1_score(y_test, (y_pred_prob >= t).astype(int)) for t in thresholds]
best_thresh = thresholds[np.argmax(f1_scores)]
 
print(f"\nBest threshold for max F1 (Diabetes): {best_thresh:.2f}")
print(f"F1 at best threshold               : {max(f1_scores):.4f}")
print(f"F1 at default threshold (0.50)     : {f1_score(y_test, (y_pred_prob >= 0.50).astype(int)):.4f}")


Best threshold for max F1 (Diabetes): 0.59
F1 at best threshold               : 0.7926
F1 at default threshold (0.50)     : 0.7583


In [21]:
y_pred_best = (y_pred_prob >= best_thresh).astype(int)
 
print("\n── Classification Report (tuned threshold) ────────────────")
print(classification_report(y_test, y_pred_best, target_names=["No Diabetes", "Diabetes"]))
 
print(f"ROC-AUC Score : {roc_auc_score(y_test, y_pred_prob):.4f}")
 
cm = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")


── Classification Report (tuned threshold) ────────────────
              precision    recall  f1-score   support

 No Diabetes       0.98      0.99      0.98     18300
    Diabetes       0.85      0.74      0.79      1700

    accuracy                           0.97     20000
   macro avg       0.92      0.86      0.89     20000
weighted avg       0.97      0.97      0.97     20000

ROC-AUC Score : 0.9785

Confusion Matrix:
  TN=18085  FP=215
  FN=443  TP=1257


In [22]:
print("\n── Threshold vs F1 (Diabetes class) ─────────────────────")
print(f"{'Threshold':>10}  {'F1':>8}  {'Recall':>8}  {'Precision':>10}")
from sklearn.metrics import precision_score, recall_score
for t in np.arange(0.25, 0.55, 0.05):
    preds = (y_pred_prob >= t).astype(int)
    print(f"  {t:.2f}       "
          f"{f1_score(y_test, preds):.4f}    "
          f"{recall_score(y_test, preds):.4f}    "
          f"{precision_score(y_test, preds):.4f}")


── Threshold vs F1 (Diabetes class) ─────────────────────
 Threshold        F1    Recall   Precision
  0.25       0.6002    0.9306    0.4429
  0.30       0.6353    0.9047    0.4895
  0.35       0.6752    0.8841    0.5461
  0.40       0.7083    0.8547    0.6047
  0.45       0.7383    0.8200    0.6715
  0.50       0.7583    0.7882    0.7306
  0.55       0.7829    0.7606    0.8066


In [26]:
import joblib

# Save the model
joblib.dump(model, "../models/diabetes_xgboost_model.pkl")
print("Model saved!")

# Save the best threshold too (important for predictions)
joblib.dump(best_thresh, "../models/best_threshold_diabetes.pkl")
print(f"Threshold saved: {best_thresh:.2f}")

Model saved!
Threshold saved: 0.59
